# Lab: Evidence, model selection, detecting changes

Write two functions described in the slides: `log_evidence` and `log_bf` and apply them.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('../..') if os.getcwd().endswith('changes') else os.getcwd())
import numpy as np
import matplotlib.pyplot as plt
from adaptation import BayesianLinearRegression
from labs.changes.robot import SIGMA, WORLDS, stream, closed_form_evidence

TAU = 1.     # prior SD of a change


def gauss_logpdf(y, mean, sd):
    return np.sum(-.5*((y - mean)/sd)**2 - np.log(sd) - .5*np.log(2*np.pi), -1)


def blr(n_features, n_outputs=1):
    return BayesianLinearRegression(n_features, n_outputs, prior_std=TAU, noise_std=SIGMA)

## 1. Evidence

Score each point with the model's predictive *before* it learns that point. The sum is the log evidence.

In [ ]:
def log_evidence(model, X, y):
    """Log predictive density of each row of y (scored before learning it); model learns X, y as it goes."""
    # TODO
    ...


a, y, r = stream(WORLDS['swap'])
X, y0 = a[100:130], r[100:130, :1]
print(log_evidence(blr(2), X, y0).sum(), closed_form_evidence(X, y0[:, 0], TAU))   # should match

## 2. What changed?

- **S0, nothing changed:** `r = noise`.
- **S1, motor strengths:** output i uses feature `a_i` only, so one 1-feature BLR per motor.
- **S2, wires mixed:** both outputs use features `a`, so one BLR with 2 features and 2 outputs.

The plot shows the posterior over S0, S1, S2 (equal prior) during the 30 steps after the change.

In [ ]:
def structure_log_evidence(a, r):
    """Log evidence of each step under S0, S1, S2: shape (len(a), 3)."""
    s0 = gauss_logpdf(r, 0, SIGMA)
    s1 = log_evidence(blr(1), a[:, :1], r[:, :1]) + log_evidence(blr(1), a[:, 1:], r[:, 1:])
    s2 = ...   # TODO
    return np.stack([s0, s1, s2], 1)


def posterior(E):
    """Posterior over S0, S1, S2 from their log evidences E (..., 3), equal prior."""
    # TODO
    ...


fig, axs = plt.subplots(1, 3, figsize=(12, 3))
for ax, (name, M) in zip(axs, WORLDS.items()):
    a, y, r = stream(M)
    E = structure_log_evidence(a[100:130], r[100:130])
    P = posterior(np.cumsum(E, 0))   # after each step
    ax.plot(P, label=['S0 nothing changed', 'S1 motor strengths', 'S2 wires mixed'])
    ax.set(title=name, xlabel='steps after the change', ylim=(-.02, 1.02))
axs[0].legend(fontsize=8); fig.tight_layout()

## 3. When did it change?

Test each candidate step s: do the next L steps fit a fresh model better than the model of the W steps
before s? Declare a change when log BF exceeds log((1 - h)/h).

In [ ]:
def log_bf(X_old, y_old, X_new, y_new):
    """log p(new | fresh model) - log p(new | model fitted on old), using the S2 model."""
    # TODO
    ...


def scan(a, r, W=50, L=10):
    """log BF for every candidate change step s."""
    bf = {}
    for s in range(W, len(a) - L + 1):
        bf[s] = log_bf(a[s - W:s], r[s - W:s], a[s:s + L], r[s:s + L])
    return bf


h = 1e-3
threshold = np.log((1 - h)/h)
for name in ('swap', 'none'):
    a, y, r = stream(WORLDS[name])
    bf = scan(a, r)
    plt.plot(list(bf.keys()), list(bf.values()), label=name)
plt.axhline(threshold, color='k', ls=':', label='threshold')
plt.axvline(100, color='0.7')
plt.ylim(-30, 100); plt.xlabel('candidate change step s'); plt.ylabel('log BF'); plt.legend();

## 4. Detect, then diagnose

Find the first step s where the test fires and choose a model

In [ ]:
for name in ('weak', 'swap'):
    a, y, r = stream(WORLDS[name])
    s = ...   # TODO: first step s whose log BF exceeds the threshold
    P = ...   # TODO: posterior over S0, S1, S2 for the 30 steps from s
    print(name, s, np.round(P, 2))

## 5. Explore

How small a change can you detect?

In [ ]:
g, L = .7, 10
found = 0
for seed in range(8):
    a, y, r = stream(np.diag([g, g]), seed=seed)
    bf = scan(a, r, L=L)
    near_change = [bf[s] for s in range(70, 131)]
    found += max(near_change) > threshold
print(f'gain {g}, L = {L}: detected in {found}/8 runs')